# 05 — Pro Consumidor Prices | Silver Transformation

One row per **month × product × presentation × retail segment**, price in DOP.

| Rule | Treatment |
|---|---|
| Year outside the report period | The title states the period (e.g. "Julio - Septiembre del 2026"). Rows with another year (2027–2041 in the current file) are **rejected** to `quality.dq_rejected_records`, never corrected by hand. |
| Month | Spanish month name must be valid and inside the report period. |
| Price | Numeric and > 0. |
| Label variants | The same product or presentation is sometimes written with different capitals or spacing ("Aceite de Soya Crisol" / "Aceite de soya Crisol", "Lata 106gr./ 95 g." / "Lata 106 gr./ 95 g."). Labels that are equal ignoring case and spaces get one canonical label (the most frequent one); the original label is kept in `source_product` / `source_presentation`. Labels with different words are never merged. |
| Duplicates | One row per month, product, presentation and segment within a file; extra copies are rejected. |
| Several report files | Bronze keeps one snapshot per archived workbook (one per quarterly report). Each file is validated against the period in its own title. When two files carry the same month × product × presentation × segment, the newest arrival wins; the older row is counted as `superseded_by_newer_file` (observational), not rejected. |

Target: `silver.proconsumidor_prices_monthly` · full refresh · **Status:** Development.

In [0]:
import re, uuid, json
from datetime import datetime, timezone, date
from pyspark.sql import functions as F

CATALOG = "rd_economic_intelligence"
SOURCE = f"{CATALOG}.bronze.proconsumidor_prices_raw"
TARGET = f"{CATALOG}.silver.proconsumidor_prices_monthly"
REJECTED = f"{CATALOG}.quality.dq_rejected_records"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
MONTHS = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

rows = spark.table(SOURCE).orderBy("_source_modified_ts", "_source_path", "_source_row_number").collect()
snapshots = list(dict.fromkeys(r["_source_path"] for r in rows))   # oldest arrival first
order = {p: i for i, p in enumerate(snapshots)}
periods = {}
for r in rows:
    if r["_source_path"] not in periods:
        m = re.search(r"([A-Za-zé]+)\s*-\s*([A-Za-zé]+)\s+del?\s+(\d{4})", r["_report_title"])
        if m is None:
            raise ValueError(f"Report period not found in title: {r['_report_title']!r} ({r['_source_path']})")
        periods[r["_source_path"]] = (int(m.group(3)), MONTHS[m.group(1).lower()], MONTHS[m.group(2).lower()], r["_report_title"], r["_source_file"])
for p in snapshots:
    print(f"Snapshot {order[p] + 1}: {p.split('consumer_prices/')[-1]} · months {periods[p][1]}-{periods[p][2]} of {periods[p][0]}")

good, rejects = [], []
for r in rows:
    report_year, m1, m2, title, source_file = periods[r["_source_path"]]
    key = f"{r['_source_path'].split('consumer_prices/')[-1]}#row{r['_source_row_number']}"
    rec = {k: r[k] for k in ["id", "producto", "tipo", "precio", "mes", "anio", "establecimiento"]}
    try:
        year = int(float(r["anio"]))
    except (TypeError, ValueError):
        rejects.append(("invalid_year", key, rec, r["_batch_id"])); continue
    if year != report_year:
        rejects.append(("year_outside_report_period", key, rec, r["_batch_id"])); continue
    month = MONTHS.get(str(r["mes"]).strip().lower())
    if month is None or not (m1 <= month <= m2):
        rejects.append(("month_outside_report_period", key, rec, r["_batch_id"])); continue
    try:
        price = float(r["precio"])
    except (TypeError, ValueError):
        rejects.append(("non_numeric_price", key, rec, r["_batch_id"])); continue
    if price <= 0:
        rejects.append(("non_positive_price", key, rec, r["_batch_id"])); continue
    good.append((date(year, month, 1), " ".join(str(r["producto"]).split()), " ".join(str(r["tipo"]).split()),
                 " ".join(str(r["establecimiento"]).split()).lower(), price, r["_source_row_number"], r["_batch_id"], r["_source_path"]))

# Canonical labels: variants equal ignoring case and spaces map to their most frequent spelling
from collections import Counter, defaultdict
def canonical(values):
    groups = defaultdict(Counter)
    for v in values: groups[re.sub(r"\s+", "", v).lower()][v] += 1
    return {v: max(c.items(), key=lambda kv: (kv[1], kv[0]))[0] for c in groups.values() for v in c}
canon_prod, canon_pres = canonical([g[1] for g in good]), canonical([g[2] for g in good])
label_variants = sum(1 for g in good if canon_prod[g[1]] != g[1] or canon_pres[g[2]] != g[2])
good = [(g[0], canon_prod[g[1]], canon_pres[g[2]], g[3], g[4], g[5], g[6], g[1], g[2], g[7]) for g in good]
print(f"Rows with a non-canonical product or presentation label: {label_variants}")

# Within a file: one row per business key, later copies rejected. Across files: the newest snapshot wins.
seen, latest = set(), {}
for g in sorted(good, key=lambda g: (order[g[9]], g[5])):
    k = (g[0], g[1], g[2], g[3])
    if (g[9],) + k in seen:
        rejects.append(("duplicate_month_product_segment", f"{g[9].split('consumer_prices/')[-1]}#row{g[5]}",
                        dict(zip(["reference_month", "product", "presentation", "retail_segment", "price_dop"], map(str, g[:5]))), g[6]))
        continue
    seen.add((g[9],) + k)
    latest[k] = g                     # iterating oldest → newest, so the newest file overwrites
superseded = len(seen) - len(latest)
df = spark.createDataFrame([g + (periods[g[9]][4], periods[g[9]][3]) for g in latest.values()],
    "reference_month date, product string, presentation string, retail_segment string, price_dop double, _source_row_number int, _bronze_batch_id string, "
    "source_product string, source_presentation string, _source_path string, _source_file string, _report_title string")
silver = (df.withColumn("price_dop", F.col("price_dop").cast("decimal(12,2)"))
          .withColumn("_silver_run_id", F.lit(RUN_ID)).withColumn("_silver_processed_ts", F.lit(RUN_TS).cast("timestamp")))

checks = []
def add(name, rtype, tested, failed):
    checks.append((RUN_ID, TARGET, name, rtype, int(tested), int(failed), "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))
total = len(rows); n = silver.count()
from collections import Counter
rc = Counter(x[0] for x in rejects)
for rule in ["year_outside_report_period", "month_outside_report_period", "non_numeric_price", "non_positive_price", "duplicate_month_product_segment", "invalid_year"]:
    add(rule, "enforced_reject", total, rc.get(rule, 0))
add("label_variants_canonicalized", "observational", len(good), label_variants)
add("superseded_by_newer_file", "observational", len(seen), superseded)
add("reconciliation_silver_plus_rejected_plus_superseded_equals_bronze", "enforced", total, abs(total - n - len(rejects) - superseded))
for c in checks: print(f"  {c[6]:<4} {c[2]:<46} failed={c[5]:,}/{c[4]:,}")
assert checks[-1][6] == "PASS"

silver.orderBy("product", "presentation", "retail_segment", "reference_month").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Silver: Pro Consumidor basic-goods prices (DOP) by month, product, presentation and retail segment, from all archived reports (newest report wins on overlap). Rows with years outside the report period are rejected.'")
if rejects:
    spark.createDataFrame([(RUN_ID, TARGET, r[0], r[1], json.dumps(r[2], ensure_ascii=False), r[3], RUN_TS) for r in rejects],
        "run_id string, target_table string, rule_name string, record_key string, record_json string, source_batch_id string, detected_ts timestamp").write.mode("append").saveAsTable(REJECTED)
spark.createDataFrame([(c[0], c[1], c[2], "enforced" if c[3] != "observational" else c[3], c[4], c[5], "PASS" if c[5] == 0 else ("WARN" if c[3] == "observational" else ("REJECTED" if c[3] == "enforced_reject" else "FAIL")), c[7]) for c in checks],
    "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)

s = spark.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT product) p, COUNT(DISTINCT retail_segment) seg, MIN(reference_month) a, MAX(reference_month) z FROM {TARGET}").first()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "bronze_rows": total, "silver_rows": s["n"], "products": s["p"], "segments": s["seg"],
    "months": [str(s["a"]), str(s["z"])], "snapshots": len(snapshots), "superseded_by_newer_file": superseded,
    "rejected": dict(rc), "label_variants_canonicalized": label_variants}, ensure_ascii=False))